In [7]:
extern crate regex;
extern crate scandir;
extern crate jwalk;
use scandir::Walk;
use std::error::Error;
use regex::Regex;
use jwalk::WalkDir;

use std::time::Instant;
use std::error::Error;

pub fn find_files(pattern: &str, directory: &str,show_hidden:bool) -> Result<(), Box<dyn Error>> {
    const PATH_PREFIX: &str = if cfg!(windows) { "" } else { "/" };
    let re: Regex = Regex::new(pattern)?;

    
  
    let walk:Vec<String>  = Walk::new(&directory, Some(true))?
            .skip_hidden(!show_hidden)
            .collect()?
            .files();
    

    for file in walk {
        if re.is_match(&file) {
            println!("{}{}", PATH_PREFIX, file);
        }
    }

    Ok(())
}





In [8]:


fn testagain(directory: &str) -> Result<(), Box<dyn Error + 'static>> {
    let start = Instant::now();
    let mut count = 0;
    let mut skipped = 0;

    for entry in WalkDir::new(directory) {
        match entry {
            Ok(_) => count += 1,
            Err(e) if e.to_string().contains("Permission denied") => {
                skipped += 1;
                continue
            },
            Err(e) => return Err(Box::new(e))
        }
    }

    println!("Found {} entries, skipped {} in {:?}", count, skipped, start.elapsed());
    Ok(())
}

testagain("/")

Found 1714308 entries, skipped 2 in 397.857868ms


Ok(())

In [9]:


fn test_with_regex(directory: &str, pattern: &str) -> Result<(), Box<dyn Error + 'static>> {
    let start = Instant::now();
    let mut count = 0;
    let mut skipped = 0;
    let re = Regex::new(pattern)?;

    for entry in WalkDir::new(directory) {
        match entry {
            Ok(e) => {
                if re.is_match(e.path().to_string_lossy().as_ref()) {
                    count += 1;
                }
            },
            Err(e) if e.to_string().contains("Permission denied") => {
                skipped += 1;
                continue
            },
            Err(e) => return Err(Box::new(e))
        }
    }

    println!("Found {} matches, skipped {} in {:?}", count, skipped, start.elapsed());
    Ok(())
}

test_with_regex("/","zshrc")

Found 1 matches, skipped 2 in 667.421202ms


Ok(())

In [10]:
use regex::Regex;

fn test_with_reg(directory: &str, pattern: &str) -> Result<(), Box<dyn Error + 'static>> {
    let start = Instant::now();
    let mut matches = Vec::new();
    let mut skipped = 0;
    let re = Regex::new(pattern)?;

    for entry in WalkDir::new(directory) {
        match entry {
            Ok(e) => {
                let path = e.path().to_string_lossy();
                if re.is_match(&path) {
                    matches.push(path.to_string());
                }
            },
            Err(e) if e.to_string().contains("Permission denied") => {
                skipped += 1;
                continue
            },
            Err(e) => return Err(Box::new(e))
        }
    }

    println!("\nMatches found: {}", matches.len());
    for path in &matches {
        println!("{}", path);
    }
    println!("\nSkipped {} entries, completed in {:?}", skipped, start.elapsed());
    
    Ok(())
}

test_with_reg("/","zshrc")

Error: temporary value dropped while borrowed

In [19]:
fn test_with_r(pattern: &str, directory: &str,show_hidden:bool) -> Result<(), Box<dyn Error + 'static>> {
    let start = Instant::now();
    let mut matches = Vec::new();
    let re = Regex::new(pattern)?;

    for entry in WalkDir::new(directory).skip_hidden(!show_hidden) {
        match entry {
            Ok(e) => {
                let path = e.path().to_string_lossy().into_owned();
                if re.is_match(&path) {
                    matches.push(path);
                }
            },
            Err(e) if e.to_string().contains("Permission denied") => {
                //skipped += 1;
                continue
            },
            Err(e) => return Err(Box::new(e))
        }
    }

    for path in &matches {
        println!("{}", path);
    }
    println!("{:?}",start.elapsed());
    
    Ok(())
}

